# 1. Import Libraries

This notebook focuses on building and evaluating machine learning models for Walmart Weekly Sales Forecasting.

The required libraries are imported for:
- Data manipulation
- Model training
- Data preprocessing
- Performance evaluation

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# 2. Load Feature Engineered Dataset

The feature-engineered dataset created in the previous notebook is loaded.

This dataset already contains:
- Date features
- Lag features
- Rolling statistics
- Exponential moving averages
- Business features
- Cyclical features

In [ ]:
df = pd.read_csv(
    "walmart_feature_engineered.csv",
    parse_dates=["Date"]
)

df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,...,Month,Week,Day,DayOfWeek,MonthStart,MonthEnd,Month_Sin,Month_Cos,Week_Sin,Week_Cos
0,1,1,2010-02-05,24924.50,0,42.31,2.572,0.0,0.0,0.0,...,2,5,5,4,0,0,0.866025,5.000000e-01,0.568065,0.822984
1,1,1,2010-02-12,46039.49,1,38.51,2.548,0.0,0.0,0.0,...,2,6,12,4,0,0,0.866025,5.000000e-01,0.663123,0.748511
2,1,1,2010-02-19,41595.55,0,39.93,2.514,0.0,0.0,0.0,...,2,7,19,4,0,0,0.866025,5.000000e-01,0.748511,0.663123
3,1,1,2010-02-26,19403.54,0,46.63,2.561,0.0,0.0,0.0,...,2,8,26,4,0,0,0.866025,5.000000e-01,0.822984,0.568065
4,1,1,2010-03-05,21827.90,0,46.50,2.625,0.0,0.0,0.0,...,3,9,5,4,0,0,1.000000,6.123234e-17,0.885456,0.464723


# 3. Missing Value Analysis

Before training machine learning models, it is important to identify missing values.

Lag features and rolling statistics naturally introduce missing values because historical observations are unavailable for the first few records of each Store–Department combination.

In [ ]:
missing = df.isnull().sum()

missing = missing[missing > 0]

missing.sort_values(ascending=False)

Lag_52            160487
Lag_12             38615
RollingMean_12     38615
RollingStd_12      38615
RollingMax_12      38615
RollingMin_12      38615
Lag_8              25966
RollingMin_8       25966
RollingMean_8      25966
RollingStd_8       25966
RollingMax_8       25966
RollingMax_4       13134
RollingStd_4       13134
RollingMean_4      13134
Lag_4              13134
RollingMin_4       13134
Lag_2               6625
ExpandingStd        6625
Lag_1               3331
ExpandingMean       3331
EMA_4               3331
EMA_8               3331
EMA_12              3331
dtype: int64

"The missing values are generated because historical observations do not exist for the initial periods after creating lag and rolling features. Filling them with zero or the mean would introduce artificial information. Therefore, those rows are removed so that every training sample has valid historical context."

# 4. Handle Missing Values

The missing values originate from lag and rolling feature engineering.

Since these missing values represent unavailable historical information rather than data collection errors, the corresponding rows are removed.

This ensures that every observation used for model training has complete historical context.

In [ ]:
print("Shape before dropping:", df.shape)

df = df.dropna().reset_index(drop=True)

print("Shape after dropping :", df.shape)

Shape before dropping: (421570, 60)
Shape after dropping : (261083, 60)


Why did you lose so many rows?"


"The project includes a 52-week lag feature to capture yearly seasonality. Since the first 52 weeks of each Store-Department combination do not have a previous year's observation, those rows contain missing values and are removed before training."

# 5. Feature Matrix and Target Variable

The target variable is **Weekly_Sales**.

The raw **Date** column is removed because all relevant temporal information has already been transformed into engineered features such as month, week, cyclical encoding, lag variables, rolling statistics, and exponential moving averages.

In [ ]:
X = df.drop(columns=["Weekly_Sales", "Date"])
y = df["Weekly_Sales"]

print("Feature Matrix Shape :", X.shape)
print("Target Shape         :", y.shape)

Feature Matrix Shape : (261083, 58)
Target Shape         : (261083,)


# 6. Chronological Train-Test Split

Unlike traditional machine learning problems, forecasting models must be trained only on historical observations and evaluated on future observations.

A chronological split prevents data leakage by ensuring that the model never learns from future information during training.

In [ ]:
# Sort by date to preserve the time order
df = df.sort_values("Date").reset_index(drop=True)

# Recreate X and y after sorting
X = df.drop(columns=["Weekly_Sales", "Date"])
y = df["Weekly_Sales"]

# 80% train, 20% test
split_index = int(len(df) * 0.8)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples :", X_train.shape)
print("Testing samples  :", X_test.shape)

Training samples : (208866, 58)
Testing samples  : (52217, 58)


I used StandardScaler because Linear Regression benefits from standardized features with zero mean and unit variance. It improves optimization stability while preserving the relative distribution of the data."

# 7. Feature Scaling

Linear Regression is sensitive to the scale of input features.

The training data is used to fit the scaler, and the same transformation is applied to the test data to avoid data leakage.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(type(X_train_scaled))
print(X_train_scaled.shape)
print(X_test_scaled.shape)

<class 'numpy.ndarray'>
(208866, 58)
(52217, 58)


StandardScaler is a numerical transformation. It performs mathematical operations on the values and returns a NumPy array instead of preserving the DataFrame structure.

In [ ]:
X_train_scaled = pd.DataFrame(
    X_train_scaled,
    columns=X_train.columns,
    index=X_train.index
)

# 8. Linear Regression (Baseline Model)

Linear Regression is used as the baseline model.

Although sales forecasting often involves nonlinear relationships, Linear Regression provides a simple benchmark against which more complex models can be compared.

In [ ]:
from sklearn.linear_model import LinearRegression

# Create the model
lr = LinearRegression()

# Train the model
lr.fit(X_train_scaled, y_train)

LinearRegression()

"fit() learns the relationship between the input features (X) and the target (y). In Linear Regression, it estimates the coefficients and intercept that minimize the sum of squared errors on the training data."

# 9. Predictions

Predictions are generated on the scaled test dataset.

The model has never seen these observations during training, allowing us to evaluate its ability to generalize to future data.

In [ ]:
y_pred_lr = lr.predict(X_test_scaled)

print(y_pred_lr[:10])

[31396.07248276 20990.9886085   6557.86958684  6681.93229505
 34609.49738562 21495.55247497  3598.02401796 15328.14562526
  8414.49690002  2623.80421169]


c:\Users\Saurya prakash\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


# 10. Model Evaluation

The Linear Regression model is evaluated using three commonly used regression metrics:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- R² Score

These metrics measure prediction accuracy from different perspectives.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(y_test, y_pred_lr)

rmse = np.sqrt(mean_squared_error(y_test, y_pred_lr))

r2 = r2_score(y_test, y_pred_lr)

print(f"MAE  : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

MAE  : 1574.12
RMSE : 2889.55
R²   : 0.9828


"RMSE is higher than MAE because it penalizes larger prediction errors more heavily. It helps identify whether the model occasionally makes large forecasting mistakes."

"An R² score of 0.9828 indicates that the Linear Regression model captures most of the variability in weekly sales, suggesting an excellent fit on the test data."

###
checking for data leakage because of this unusual hugh score by a lr model.

In [ ]:
print(X.columns.tolist())

['Store', 'Dept', 'IsHoliday', 'Temperature', 'Fuel_Price', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5', 'CPI', 'Unemployment', 'Size', 'Lag_1', 'Lag_2', 'Lag_4', 'Lag_8', 'Lag_12', 'Lag_52', 'RollingMean_4', 'RollingStd_4', 'RollingMax_4', 'RollingMin_4', 'RollingMean_8', 'RollingStd_8', 'RollingMax_8', 'RollingMin_8', 'RollingMean_12', 'RollingStd_12', 'RollingMax_12', 'RollingMin_12', 'ExpandingMean', 'ExpandingStd', 'EMA_4', 'EMA_8', 'EMA_12', 'StoreType_B', 'StoreType_C', 'MarkDown1_Missing', 'MarkDown2_Missing', 'MarkDown3_Missing', 'MarkDown4_Missing', 'MarkDown5_Missing', 'MarkDown_Total', 'Holiday_Promotion', 'Active_MarkDown_Count', 'Year', 'Quarter', 'Month', 'Week', 'Day', 'DayOfWeek', 'MonthStart', 'MonthEnd', 'Month_Sin', 'Month_Cos', 'Week_Sin', 'Week_Cos']


# 11. Ablation Study

An ablation study is performed to measure the contribution of historical time-series features.

All lag, rolling, expanding, and exponential moving average (EMA) features are removed, and the Linear Regression model is retrained.

Comparing the performance before and after removal helps quantify the predictive value of these engineered historical features.

In [ ]:
history_features = [
    "Lag_1", "Lag_2", "Lag_4", "Lag_8", "Lag_12", "Lag_52",

    "RollingMean_4", "RollingStd_4", "RollingMax_4", "RollingMin_4",
    "RollingMean_8", "RollingStd_8", "RollingMax_8", "RollingMin_8",
    "RollingMean_12", "RollingStd_12", "RollingMax_12", "RollingMin_12",

    "ExpandingMean", "ExpandingStd",

    "EMA_4", "EMA_8", "EMA_12"
]

X_ablation = X.drop(columns=history_features)

print("Original Features :", X.shape[1])
print("Remaining Features:", X_ablation.shape[1])

Original Features : 58
Remaining Features: 35


In [ ]:
X_ablation.head()

,Store,Dept,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,...,Month,Week,Day,DayOfWeek,MonthStart,MonthEnd,Month_Sin,Month_Cos,Week_Sin,Week_Cos
0,1,1,0,42.27,2.989,0.0,0.0,0.0,0.0,0.0,...,2,5,4,4,0,0,0.866025,0.5,0.568065,0.822984
1,9,30,0,31.82,2.989,0.0,0.0,0.0,0.0,0.0,...,2,5,4,4,0,0,0.866025,0.5,0.568065,0.822984
2,38,80,0,45.14,3.348,0.0,0.0,0.0,0.0,0.0,...,2,5,4,4,0,0,0.866025,0.5,0.568065,0.822984
3,9,29,0,31.82,2.989,0.0,0.0,0.0,0.0,0.0,...,2,5,4,4,0,0,0.866025,0.5,0.568065,0.822984
4,9,28,0,31.82,2.989,0.0,0.0,0.0,0.0,0.0,...,2,5,4,4,0,0,0.866025,0.5,0.568065,0.822984


In [ ]:
X_train_ab = X_ablation.iloc[:split_index]
X_test_ab = X_ablation.iloc[split_index:]

In [ ]:
scaler_ab = StandardScaler()

X_train_ab = scaler_ab.fit_transform(X_train_ab)

X_test_ab = scaler_ab.transform(X_test_ab)

In [ ]:
lr_ab = LinearRegression()

lr_ab.fit(X_train_ab, y_train)

LinearRegression()

In [ ]:
y_pred_ab = lr_ab.predict(X_test_ab)

In [ ]:
mae_ab = mean_absolute_error(y_test, y_pred_ab)

rmse_ab = np.sqrt(mean_squared_error(y_test, y_pred_ab))

r2_ab = r2_score(y_test, y_pred_ab)

print(f"MAE  : {mae_ab:.2f}")
print(f"RMSE : {rmse_ab:.2f}")
print(f"R²   : {r2_ab:.4f}")

MAE  : 14446.31
RMSE : 20933.16
R²   : 0.0976


"How do you know your model isn't leaking data?"


"I performed an ablation study by removing all lag, rolling, expanding, and EMA features. The R² dropped from 0.9828 to 0.0976, demonstrating that the predictive performance comes from legitimate historical information rather than accidental leakage. I also verified that all historical features were created using only past observations via shift(1), and the train-test split was chronological."

"How do you know your model isn't leaking data?"

A strong answer would be:

"I performed an ablation study by removing all lag, rolling, expanding, and EMA features. The R² dropped from 0.9828 to 0.0976, demonstrating that the predictive performance comes from legitimate historical information rather than accidental leakage. I also verified that all historical features were created using only past observations via shift(1), and the train-test split was chronological."

# 12. Random Forest Regression

Random Forest is an ensemble learning algorithm that combines multiple decision trees.

Unlike Linear Regression, Random Forest can capture nonlinear relationships and complex feature interactions.

Feature scaling is not required because decision trees split data using thresholds rather than distance calculations.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

RandomForestRegressor(n_jobs=-1, random_state=42)

In [ ]:
y_pred_rf = rf.predict(X_test)

In [ ]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

r2_rf = r2_score(y_test, y_pred_rf)

print(f"MAE  : {mae_rf:.2f}")
print(f"RMSE : {rmse_rf:.2f}")
print(f"R²   : {r2_rf:.4f}")

MAE  : 1216.21
RMSE : 2516.91
R²   : 0.9870


Random Forest only improved R² from 0.9828 to 0.9870. Is it worth using?"

"I wouldn't decide based on R² alone. Random Forest reduced both MAE and RMSE, meaning the average prediction error and large prediction errors both decreased. I would compare all metrics along with training time, inference speed, and interpretability before choosing the final model."

In [ ]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


# 13. XGBoost Regression

XGBoost (Extreme Gradient Boosting) is an ensemble algorithm that builds decision trees sequentially.

Each new tree attempts to correct the prediction errors made by the previous trees, making XGBoost one of the most powerful algorithms for structured tabular datasets.

In [ ]:
from xgboost import XGBRegressor

xgb = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

xgb.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.1, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=-1, num_parallel_tree=None, ...)

In [ ]:
y_pred_xgb = xgb.predict(X_test)

In [ ]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)

rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

r2_xgb = r2_score(y_test, y_pred_xgb)

print(f"MAE  : {mae_xgb:.2f}")
print(f"RMSE : {rmse_xgb:.2f}")
print(f"R²   : {r2_xgb:.4f}")

MAE  : 1241.59
RMSE : 2591.05
R²   : 0.9862


Why didn't XGBoost perform better?"

A strong answer is:

"XGBoost often benefits from careful hyperparameter tuning. In this project, both models were trained with baseline settings. Since the dataset already contains strong lag and rolling features, Random Forest performed slightly better without additional tuning. I would perform hyperparameter optimization before drawing final conclusions."

In [ ]:
import lightgbm

print(lightgbm.__version__)

4.7.0


# 14. LightGBM Regression

LightGBM is a gradient boosting framework developed by Microsoft.

Unlike XGBoost, which grows trees level-wise, LightGBM grows trees leaf-wise, allowing it to train faster and often achieve better accuracy on large structured datasets.

In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

lgbm.fit(X_train, y_train)

c:\Users\Saurya prakash\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\Saurya prakash\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\Users\Saurya prakash\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Saurya prakash\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.056016 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 8869
[LightGBM] [Info] Number of data points in the train set: 208866, number of used features: 57
[LightGBM] [Info] Start training from score 16500.391521
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


LGBMRegressor(max_depth=6, random_state=42)

In [ ]:
y_pred_lgbm = lgbm.predict(X_test)

In [ ]:
mae_lgbm = mean_absolute_error(y_test, y_pred_lgbm)

rmse_lgbm = np.sqrt(mean_squared_error(y_test, y_pred_lgbm))

r2_lgbm = r2_score(y_test, y_pred_lgbm)

print(f"MAE  : {mae_lgbm:.2f}")
print(f"RMSE : {rmse_lgbm:.2f}")
print(f"R²   : {r2_lgbm:.4f}")

MAE  : 1268.50
RMSE : 2642.12
R²   : 0.9856


"Your LightGBM performed worse than Random Forest. Does that mean LightGBM is a bad algorithm?"

A good answer is:

"No. Model performance is dataset-dependent. In this project, Random Forest achieved better results with baseline hyperparameters. LightGBM often requires parameter tuning to realize its full potential. Therefore, I would not conclude that LightGBM is inferior without a proper hyperparameter optimization study."

# 15. CatBoost Regression

CatBoost is a gradient boosting algorithm developed by Yandex.

It is specifically designed to handle categorical features effectively and often performs well on structured tabular datasets while requiring minimal preprocessing.

In [ ]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from catboost import CatBoostRegressor

cat = CatBoostRegressor(
    iterations=100,
    learning_rate=0.1,
    depth=6,
    random_seed=42,
    verbose=0
)

cat.fit(X_train, y_train)

CatBoostRegressor(depth=6, iterations=100, learning_rate=0.1, loss_function='RMSE', random_seed=42, verbose=0)

In [ ]:
y_pred_cat = cat.predict(X_test)

In [ ]:
mae_cat = mean_absolute_error(y_test, y_pred_cat)

rmse_cat = np.sqrt(mean_squared_error(y_test, y_pred_cat))

r2_cat = r2_score(y_test, y_pred_cat)

print(f"MAE  : {mae_cat:.2f}")
print(f"RMSE : {rmse_cat:.2f}")
print(f"R²   : {r2_cat:.4f}")

MAE  : 1392.32
RMSE : 2844.26
R²   : 0.9833


In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "MAE": [
        mae,
        mae_rf,
        mae_xgb,
        mae_lgbm,
        mae_cat
    ],
    "RMSE": [
        rmse,
        rmse_rf,
        rmse_xgb,
        rmse_lgbm,
        rmse_cat
    ],
    "R2 Score": [
        r2,
        r2_rf,
        r2_xgb,
        r2_lgbm,
        r2_cat
    ]
})

comparison = comparison.sort_values("R2 Score", ascending=False)

comparison.reset_index(drop=True, inplace=True)

comparison

,Model,MAE,RMSE,R2 Score
0,Random Forest,1216.208566,2516.913640,0.986955
1,XGBoost,1241.589020,2591.054662,0.986175
2,LightGBM,1268.497394,2642.115271,0.985624
3,CatBoost,1392.317431,2844.262194,0.983341
4,Linear Regression,1574.115835,2889.552275,0.982806


# 13. Hyperparameter Tuning - Random Forest

The baseline Random Forest model performed best among all baseline models.

To further improve performance, RandomizedSearchCV is used to search for better hyperparameters.

RandomizedSearchCV is preferred over GridSearchCV because it explores the parameter space more efficiently while significantly reducing computational cost.